In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from datetime import datetime , timedelta

In [4]:
# Load the dataset into a Pandas DataFrame and display the first 5 rows.
df = pd.read_csv('credit_card_customer.csv')
df.head(5)

,Transaction_ID,Customer_ID,Transaction_Date,Transaction_Type,Merchant,Category,Amount,Payment_Mode,Transaction_Status,Location,Transaction_Time
0,100000,4452.0,2023-01-01,NaN,Walmart,Travel,4520.70,Debit Card,Approved,Jonesport,NaN
1,100001,2775.0,2023-01-01,ATM,BestBuy,Travel,1437.85,Debit Card,Approved,NaN,NaN
2,100002,2259.0,2023-01-01,Mobile Payment,Uber,Clothing,3320.52,PayPal,Approved,Port James,NaN
3,100003,4545.0,2023-01-01,Online,BestBuy,Travel,2659.96,Debit Card,Approved,Hawkinston,NaN
4,100004,2137.0,2023-01-01,ATM,NaN,Travel,2517.07,Debit Card,Approved,Matthewland,NaN


In [5]:
# Check the shape, column names, and summary statistics of the dataset.
print(df.shape)
# columns name 
print(df.columns.to_list())
# summary 
print(df.describe())

(21500, 11)
['Transaction_ID', 'Customer_ID', 'Transaction_Date', 'Transaction_Type', 'Merchant', 'Category', 'Amount', 'Payment_Mode', 'Transaction_Status', 'Location', 'Transaction_Time']
             Amount
count  20100.000000
mean   13295.924845
std    14672.634153
min        6.550000
25%     2615.200000
50%     5614.275000
75%    22678.175000
max    49998.010000


In [6]:
# Identify and handle missing values (fill or drop based on the data type).
print(df.isnull().sum())

Transaction_ID            0
Customer_ID            1333
Transaction_Date       1369
Transaction_Type       1368
Merchant               1735
Category               1745
Amount                 1400
Payment_Mode           1713
Transaction_Status     1787
Location               1325
Transaction_Time      16500
dtype: int64


In [7]:
df.dtypes

Transaction_ID         object
Customer_ID            object
Transaction_Date       object
Transaction_Type       object
Merchant               object
Category               object
Amount                float64
Payment_Mode           object
Transaction_Status     object
Location               object
Transaction_Time       object
dtype: object

In [8]:
df['Transaction_Type' ] = df['Transaction_Type'].fillna('Unknown')
df['Amount'] = df['Amount'].fillna(df['Amount'].median())
print(df.isnull().sum())

Transaction_ID            0
Customer_ID            1333
Transaction_Date       1369
Transaction_Type          0
Merchant               1735
Category               1745
Amount                    0
Payment_Mode           1713
Transaction_Status     1787
Location               1325
Transaction_Time      16500
dtype: int64


In [9]:
df.dropna(subset=['Customer_ID'], inplace=True)
df.dropna(subset=['Transaction_Date'], inplace=True)
print(df.isnull().sum())

Transaction_ID            0
Customer_ID               0
Transaction_Date          0
Transaction_Type          0
Merchant               1513
Category               1537
Amount                    0
Payment_Mode           1488
Transaction_Status     1573
Location               1115
Transaction_Time      13902
dtype: int64


In [10]:
df['Merchant'] = df['Merchant'].fillna('Unknown')
df['Category'] = df['Category'].fillna('Unknown')
df['Payment_Mode'] = df['Payment_Mode'].fillna('Unknown')
df['Transaction_Status'] = df['Transaction_Status'].fillna('Unknown')
df['Location'] = df['Location'].fillna('Unknown')
print(df.isnull().sum())

Transaction_ID            0
Customer_ID               0
Transaction_Date          0
Transaction_Type          0
Merchant                  0
Category                  0
Amount                    0
Payment_Mode              0
Transaction_Status        0
Location                  0
Transaction_Time      13902
dtype: int64


In [11]:
mode_time = df["Transaction_Time"].mode()[0]

df["Transaction_Time"] = df["Transaction_Time"].fillna(mode_time)
print(df.isnull().sum())

Transaction_ID        0
Customer_ID           0
Transaction_Date      0
Transaction_Type      0
Merchant              0
Category              0
Amount                0
Payment_Mode          0
Transaction_Status    0
Location              0
Transaction_Time      0
dtype: int64


In [12]:
# Categorize the Transaction_Amount into Low, Medium, and High based on:
# Low: Below $1000
# Medium: Between $1000 - $5000
# High: Above $5000

df.head()

df['Amount_category'] = pd.cut(df['Amount'],
                      bins=[-float('inf'),1000,5000,float('inf')],
                      labels=["low","Medium","High"])

print(df[['Amount','Amount_category']].tail())

        Amount Amount_category
21495  4924.53          Medium
21496  1386.46          Medium
21497  4738.12          Medium
21498  3155.45          Medium
21499  2957.13          Medium


In [13]:
# Create a new column Discounted_Amount, assuming a 5% discount on all transactions above $500.

df['Discounted_Amount'] = df['Amount'].apply(
    lambda x : x * 0.95 if x > 5000 else x
)
print(df[['Amount','Discounted_Amount']].head())

    Amount  Discounted_Amount
0  4520.70            4520.70
1  1437.85            1437.85
2  3320.52            3320.52
3  2659.96            2659.96
4  2517.07            2517.07


In [14]:
# Find customers who made more than 10 transactions in a single day (potential fraud).
# Identify transactions that have the same Customer_ID but occurred in different locations within 5 minutes.
# Find transactions where Amount > $5000 and Transaction_Type is Online (flag as high-risk).


df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date'],errors='coerce')
print(df.dtypes)

Transaction_ID                object
Customer_ID                   object
Transaction_Date      datetime64[ns]
Transaction_Type              object
Merchant                      object
Category                      object
Amount                       float64
Payment_Mode                  object
Transaction_Status            object
Location                      object
Transaction_Time              object
Amount_category             category
Discounted_Amount            float64
dtype: object


In [15]:
df['Date'] = df['Transaction_Date'].dt.date

Transaction_per_day = df.groupby(['Customer_ID','Date']).size().reset_index(name='Transaction_Count')
print(Transaction_per_day)

      Customer_ID        Date  Transaction_Count
0          1001.0  2023-02-01                  1
1          1001.0  2023-04-15                  1
2          1003.0  2023-02-08                  1
3          1003.0  2023-06-22                  1
4          1006.0  2023-03-16                  1
...           ...         ...                ...
13874   CUST99939  2023-08-07                  1
13875   CUST99951  2023-01-30                  1
13876   CUST99985  2023-01-14                  1
13877   CUST99992  2023-07-19                  1
13878   CUST99994  2023-05-19                  1

[13879 rows x 3 columns]


In [16]:
df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date'])
df['Date'] = df['Transaction_Date'].dt.date

Transaction_per_day = (
    df.groupby(['Customer_ID', 'Date'])
      .size()
      .reset_index(name='Transaction_Count')
)

potential_fraud_customer = Transaction_per_day[
    Transaction_per_day['Transaction_Count'] > 5
]

print(potential_fraud_customer)

Empty DataFrame
Columns: [Customer_ID, Date, Transaction_Count]
Index: []


In [17]:
fraud_customers = (
    Transaction_per_day[Transaction_per_day['Transaction_Count'] > 5]
    ['Customer_ID']
    .unique()
)

fraud_transactions = df[df['Customer_ID'].isin(fraud_customers)]

print(fraud_transactions)

Empty DataFrame
Columns: [Transaction_ID, Customer_ID, Transaction_Date, Transaction_Type, Merchant, Category, Amount, Payment_Mode, Transaction_Status, Location, Transaction_Time, Amount_category, Discounted_Amount, Date]
Index: []


In [18]:
# Convert to datetime
df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date'])

# Self join on Customer_ID
merged = df.merge(df, on='Customer_ID', suffixes=('_1', '_2'))

# Filter conditions
result = merged[
    (merged['Transaction_ID_1'] < merged['Transaction_ID_2']) &
    (merged['Location_1'] != merged['Location_2']) &
    ((merged['Transaction_Date_1'] - merged['Transaction_Date_2']).abs().dt.total_seconds() <= 5000)
]

# Display required columns
print(result[[
    'Customer_ID',
    'Transaction_ID_1',
    'Location_1',
    'Transaction_Date_1',
    'Transaction_ID_2',
    'Location_2',
    'Transaction_Date_2'
]])

      Customer_ID Transaction_ID_1         Location_1 Transaction_Date_1  \
378        4507.0           100160          Bryanland         2023-01-07   
500        1910.0           100194   Lake Matthewberg         2023-01-09   
858        3094.0           100316           Ericland         2023-01-14   
1863       1241.0           100755         Dianahaven         2023-02-01   
2628       4238.0           101037  West Michaelhaven         2023-02-13   
3769       4043.0           101539         New Stacey         2023-03-06   
4971       4143.0           101956    Lake Hannahfort         2023-03-23   
5855       4635.0           102299     Port Hannahton         2023-04-06   
6075       4303.0           102388       Port Phyllis         2023-04-10   
6361       3131.0           102497           New John         2023-04-15   
6541       1776.0           102581          Jamesfurt         2023-04-18   
6604       1956.0           102616     North Kimberly         2023-04-20   
8103       4

In [19]:
customer_info = pd.read_csv("customer_info.csv")
customer_info.head()

,Customer_ID,Age,Gender,Membership_Level
0,1001,58,Male,Silver
1,1002,35,Male,Silver
2,1003,65,Male,Gold
3,1004,52,Male,Gold
4,1005,20,Male,Silver


In [27]:
# convert into int
customer_info['Customer_ID'] = df['Customer_ID'].astype(str)


In [28]:
customer_info.dtypes

Customer_ID         object
Age                  int64
Gender              object
Membership_Level    object
dtype: object

In [29]:
merged_df = pd.merge(df , customer_info , on='Customer_ID' , how="inner")
merged_df.head()

,Transaction_ID,Customer_ID,Transaction_Date,Transaction_Type,Merchant,Category,Amount,Payment_Mode,Transaction_Status,Location,Transaction_Time,Amount_category,Discounted_Amount,Date,Age,Gender,Membership_Level
0,100000,4452.0,2023-01-01,Unknown,Walmart,Travel,4520.70,Debit Card,Approved,Jonesport,06:23:46,Medium,4520.70,2023-01-01,58,Male,Silver
1,100001,2775.0,2023-01-01,ATM,BestBuy,Travel,1437.85,Debit Card,Approved,Unknown,06:23:46,Medium,1437.85,2023-01-01,35,Male,Silver
2,100002,2259.0,2023-01-01,Mobile Payment,Uber,Clothing,3320.52,PayPal,Approved,Port James,06:23:46,Medium,3320.52,2023-01-01,65,Male,Gold
3,100003,4545.0,2023-01-01,Online,BestBuy,Travel,2659.96,Debit Card,Approved,Hawkinston,06:23:46,Medium,2659.96,2023-01-01,52,Male,Gold
4,100004,2137.0,2023-01-01,ATM,Unknown,Travel,2517.07,Debit Card,Approved,Matthewland,06:23:46,Medium,2517.07,2023-01-01,20,Male,Silver


In [30]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5084 entries, 0 to 5083
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Transaction_ID      5084 non-null   object        
 1   Customer_ID         5084 non-null   object        
 2   Transaction_Date    4284 non-null   datetime64[ns]
 3   Transaction_Type    5084 non-null   object        
 4   Merchant            5084 non-null   object        
 5   Category            5084 non-null   object        
 6   Amount              5084 non-null   float64       
 7   Payment_Mode        5084 non-null   object        
 8   Transaction_Status  5084 non-null   object        
 9   Location            5084 non-null   object        
 10  Transaction_Time    5084 non-null   object        
 11  Amount_category     5084 non-null   category      
 12  Discounted_Amount   5084 non-null   float64       
 13  Date                4284 non-null   object      

In [32]:
# Find the average transaction amount per Age group.

age_bins = [0,18,30,50,100]
age_labels = ['<18','18-30','31-50','>50']
merged_df['Age_group'] = pd.cut(merged_df['Age'] , bins=age_bins, labels=age_labels,include_lowest=True)
merged_df.head()

,Transaction_ID,Customer_ID,Transaction_Date,Transaction_Type,Merchant,Category,Amount,Payment_Mode,Transaction_Status,Location,Transaction_Time,Amount_category,Discounted_Amount,Date,Age,Gender,Membership_Level,Age_group
0,100000,4452.0,2023-01-01,Unknown,Walmart,Travel,4520.70,Debit Card,Approved,Jonesport,06:23:46,Medium,4520.70,2023-01-01,58,Male,Silver,>50
1,100001,2775.0,2023-01-01,ATM,BestBuy,Travel,1437.85,Debit Card,Approved,Unknown,06:23:46,Medium,1437.85,2023-01-01,35,Male,Silver,31-50
2,100002,2259.0,2023-01-01,Mobile Payment,Uber,Clothing,3320.52,PayPal,Approved,Port James,06:23:46,Medium,3320.52,2023-01-01,65,Male,Gold,>50
3,100003,4545.0,2023-01-01,Online,BestBuy,Travel,2659.96,Debit Card,Approved,Hawkinston,06:23:46,Medium,2659.96,2023-01-01,52,Male,Gold,>50
4,100004,2137.0,2023-01-01,ATM,Unknown,Travel,2517.07,Debit Card,Approved,Matthewland,06:23:46,Medium,2517.07,2023-01-01,20,Male,Silver,18-30


In [35]:
avg_amount_per_age_group = merged_df.groupby('Age_group')['Amount'].mean().reset_index()
print(avg_amount_per_age_group)

  Age_group       Amount
0       <18  3107.860177
1     18-30  3039.425046
2     31-50  2945.729208
3       >50  3017.468112


/var/folders/f3/3kqpjfqj03s00nxsf6rppv280000gn/T/ipykernel_23117/2241725068.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  avg_amount_per_age_group = merged_df.groupby('Age_group')['Amount'].mean().reset_index()
